# 7.3 GPU 的使用

本 notebook 介绍如何在 PyTorch 中使用 GPU 加速训练，包括设备管理、数据迁移、多 GPU 训练等核心内容

### 学习目标
- 理解 CPU 与 GPU 的架构差异
- 掌握设备检测与数据迁移
- 了解 Tensor.to() 与 Model.to() 的区别
- 学会使用 DataParallel 进行多 GPU 训练
- 掌握 cudnn 相关设置

In [ ]:
import torch
import torch.nn as nn
import os

print(f"PyTorch 版本: {torch.__version__}")

## 1. CPU vs GPU 架构差异

| 特性 | CPU | GPU |
|------|-----|-----|
| 核心数 | 几个到几十个 | 数千个 |
| 单核性能 | 强 | 弱 |
| 并行能力 | 弱 | 强 |
| 适合任务 | 复杂逻辑、串行计算 | 大规模并行计算 |
| 内存 | 大（几十到几百 GB） | 小（几到几十 GB） |

**深度学习为什么适合 GPU？**
- 矩阵运算高度并行
- 同一运算应用于大量数据
- GPU 拥有大量计算单元，天然适合这种工作模式

## 2. 设备检测与设置

PyTorch 支持三种计算设备：
- `cpu`: CPU 计算
- `cuda`: NVIDIA GPU 计算
- `mps`: Apple Silicon GPU 计算（macOS）

In [ ]:
# 检测可用设备
print("=== 设备检测 ===")

# CUDA (NVIDIA GPU)
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA 版本: {torch.version.cuda}")
    print(f"  GPU 数量: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# MPS (Apple Silicon)
mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
print(f"\nMPS 是否可用: {mps_available}")

# cuDNN
print(f"\ncuDNN 是否可用: {torch.backends.cudnn.is_available()}")
if torch.backends.cudnn.is_available():
    print(f"  cuDNN 版本: {torch.backends.cudnn.version()}")

In [ ]:
# 推荐的设备选择方式
def get_device() -> torch.device:
    """自动检测最佳计算设备

    Returns:
        最佳可用设备
    """
    if torch.cuda.is_available():
        device = torch.device('cuda:0')
        print(f"使用 CUDA: {torch.cuda.get_device_name(0)}")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
        print("使用 Apple MPS")
    else:
        device = torch.device('cpu')
        print("使用 CPU")
    return device


device = get_device()
print(f"\n当前设备: {device}")

## 3. 数据迁移

### 核心区别
- `Tensor.to(device)`: **非原地操作**，返回新张量，原张量不变
- `Model.to(device)`: **原地操作**，直接修改模型，不需要重新赋值

**这是一个非常重要的区别，很多初学者在这里犯错！**

In [ ]:
# === Tensor.to() 是非原地操作 ===
x_cpu = torch.randn(3, 4)
print(f"原始张量设备: {x_cpu.device}")

# 错误用法：忘记接收返回值
x_cpu.to(device)  # 这样做没有效果！x_cpu 仍在 CPU 上
print(f"x_cpu.to(device) 后: {x_cpu.device}  <- 仍然在 CPU！")

# 正确用法：接收返回值
x_gpu = x_cpu.to(device)
print(f"x_gpu = x_cpu.to(device) 后: {x_gpu.device}  <- 正确迁移")
print(f"x_cpu 仍在: {x_cpu.device}  <- 原张量不变")

In [ ]:
# === Model.to() 是原地操作 ===
model = nn.Linear(10, 5)
print(f"模型初始设备: {next(model.parameters()).device}")

# 原地操作：不需要重新赋值
model.to(device)
print(f"model.to(device) 后: {next(model.parameters()).device}  <- 已经迁移")

In [ ]:
# .to() 的其他用法
x = torch.randn(3, 4)

# 修改设备
x1 = x.to(device)
print(f"迁移到 {device}: {x1.device}")

# 修改数据类型
x2 = x.to(torch.float16)
print(f"转为 float16: {x2.dtype}")

# 同时修改设备和数据类型
x3 = x.to(device=device, dtype=torch.float16)
print(f"迁移到 {device} + float16: device={x3.device}, dtype={x3.dtype}")

# .cuda() 和 .cpu() 快捷方式（仅限 CUDA）
if torch.cuda.is_available():
    x4 = x.cuda()     # 等价于 x.to('cuda')
    x5 = x4.cpu()     # 等价于 x.to('cpu')
    print(f"\n.cuda(): {x4.device}")
    print(f".cpu(): {x5.device}")
else:
    print("\n.cuda()/.cpu() 快捷方式需要 NVIDIA GPU")

## 4. 常用 CUDA 函数

以下函数在 CUDA 可用时非常有用。即使没有 GPU，了解这些函数也很重要

In [ ]:
print("=== CUDA 常用函数 ===")

# 1. 设备数量
print(f"\n1. torch.cuda.device_count(): {torch.cuda.device_count()}")

# 2. 当前设备
if torch.cuda.is_available():
    print(f"2. torch.cuda.current_device(): {torch.cuda.current_device()}")

    # 3. 设备名称
    print(f"3. torch.cuda.get_device_name(0): {torch.cuda.get_device_name(0)}")

    # 4. GPU 内存信息
    free_mem, total_mem = torch.cuda.mem_get_info(0)
    print(f"4. GPU 内存:")
    print(f"   总内存: {total_mem / 1024**3:.2f} GB")
    print(f"   可用内存: {free_mem / 1024**3:.2f} GB")
    print(f"   已用内存: {(total_mem - free_mem) / 1024**3:.2f} GB")

    # 5. PyTorch 分配的内存
    print(f"5. PyTorch 已分配内存: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"   PyTorch 缓存内存: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")

    # 6. 清空缓存
    torch.cuda.empty_cache()
    print(f"6. torch.cuda.empty_cache() 执行完毕")
    print("   说明: 释放未使用的 GPU 内存缓存，但不释放已分配的内存")
else:
    print("\n未检测到 CUDA GPU，以上函数仅在 NVIDIA GPU 环境中可用")
    print("\n常用函数列表:")
    print("  torch.cuda.device_count()     # GPU 数量")
    print("  torch.cuda.current_device()   # 当前 GPU 编号")
    print("  torch.cuda.get_device_name(i) # GPU 名称")
    print("  torch.cuda.mem_get_info(i)    # GPU 内存信息")
    print("  torch.cuda.memory_allocated()  # PyTorch 已分配内存")
    print("  torch.cuda.empty_cache()       # 清空未使用的缓存")

## 5. cuDNN 设置

cuDNN 是 NVIDIA 的深度学习加速库，PyTorch 内部使用它来加速卷积等操作

### 两个重要设置

| 设置 | 作用 | 推荐值 |
|------|------|--------|
| `cudnn.benchmark` | 自动选择最快的卷积算法 | 训练时 True |
| `cudnn.deterministic` | 保证结果可重复 | 需要可复现时 True |

In [ ]:
# cuDNN 设置
print("=== cuDNN 设置 ===")

# benchmark 模式：自动搜索最快的卷积算法
# 适合输入尺寸固定的场景
# 首次运行会慢一些（在搜索最优算法），之后会加速
torch.backends.cudnn.benchmark = True
print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")
print("  -> True: 自动搜索最快算法（输入尺寸固定时推荐）")
print("  -> False: 使用默认算法")

# deterministic 模式：保证结果可重复
# 会选择确定性算法，可能牺牲一些性能
torch.backends.cudnn.deterministic = True
print(f"\ncudnn.deterministic: {torch.backends.cudnn.deterministic}")
print("  -> True: 保证可重复性（实验对比时推荐）")
print("  -> False: 允许非确定性算法（更快）")

print("\n典型配置:")
print("  实验调参: benchmark=False, deterministic=True")
print("  正式训练: benchmark=True, deterministic=False")

## 6. DataParallel：多 GPU 训练

`nn.DataParallel` 是 PyTorch 最简单的多 GPU 训练方式

### 工作原理（4步）

```
1. Scatter: 将 mini-batch 均分到各 GPU
2. Replicate: 将模型复制到各 GPU
3. Parallel Forward: 各 GPU 独立前向传播
4. Gather: 将各 GPU 的输出汇总到主 GPU
```

### 注意事项
- 主 GPU (device_ids[0]) 负责汇总，内存压力更大
- 适合简单的数据并行场景
- 对于更复杂的需求，推荐使用 DistributedDataParallel

In [ ]:
class DemoModel(nn.Module):
    """演示模型"""

    def __init__(self, input_dim: int = 20, output_dim: int = 5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# DataParallel 的使用
model = DemoModel()
print(f"原始模型: {type(model).__name__}")

# 检查是否有多个 GPU
gpu_count = torch.cuda.device_count()
print(f"GPU 数量: {gpu_count}")

if gpu_count > 1:
    # 使用 DataParallel 包装模型
    model = nn.DataParallel(model)
    print(f"DataParallel 模型: {type(model).__name__}")
    print(f"使用的 GPU: {model.device_ids}")
elif gpu_count == 1:
    model = model.cuda()
    print("单 GPU 模式")
else:
    print("CPU 模式（无 GPU 可用）")

print(f"\n模型参数设备: {next(model.parameters()).device}")

In [ ]:
# DataParallel 的 forward 示例
if torch.cuda.is_available() and gpu_count > 1:
    # 创建数据
    x = torch.randn(32, 20).cuda()  # batch_size=32
    print(f"输入形状: {x.shape}")
    print(f"每个 GPU 处理: {32 // gpu_count} 个样本")

    # 前向传播（自动分发到多个 GPU）
    output = model(x)
    print(f"输出形状: {output.shape}")
    print(f"输出设备: {output.device}  (汇总到主 GPU)")
else:
    # CPU 上的演示
    x = torch.randn(32, 20)
    output = model(x)
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {output.shape}")
    print("\n多 GPU DataParallel 使用方法:")
    print("  model = nn.DataParallel(model)")
    print("  model = model.cuda()")
    print("  output = model(x.cuda())")

## 7. "module." 前缀问题

使用 `DataParallel` 后，模型的 state_dict 中所有 key 都会加上 `"module."` 前缀。这在保存和加载模型时会造成问题

In [ ]:
# 演示 module. 前缀问题
model_single = DemoModel()
print("普通模型的 state_dict key:")
for key in model_single.state_dict().keys():
    print(f"  {key}")

# 模拟 DataParallel 包装后的 key
model_dp = nn.DataParallel(DemoModel())
print("\nDataParallel 模型的 state_dict key:")
for key in model_dp.state_dict().keys():
    print(f"  {key}  <- 注意 'module.' 前缀")

In [ ]:
# 解决方案1: 保存时去掉 module. 前缀
def save_model_without_module_prefix(model: nn.Module, path: str) -> None:
    """保存模型，自动去掉 DataParallel 的 module. 前缀

    Args:
        model: 模型（可能被 DataParallel 包装）
        path: 保存路径
    """
    if isinstance(model, nn.DataParallel):
        # 通过 .module 访问原始模型
        torch.save(model.module.state_dict(), path)
    else:
        torch.save(model.state_dict(), path)


# 解决方案2: 加载时手动去掉 module. 前缀
def load_state_dict_flexible(model: nn.Module, state_dict: dict) -> None:
    """灵活加载 state_dict，自动处理 module. 前缀

    Args:
        model: 目标模型
        state_dict: 要加载的 state_dict
    """
    # 检查是否有 module. 前缀
    has_module_prefix = any(k.startswith('module.') for k in state_dict.keys())

    if has_module_prefix:
        # 去掉 module. 前缀
        new_state_dict = {}
        for k, v in state_dict.items():
            name = k.replace('module.', '', 1)  # 只替换第一个
            new_state_dict[name] = v
        model.load_state_dict(new_state_dict)
        print("已去掉 module. 前缀并加载")
    else:
        model.load_state_dict(state_dict)
        print("直接加载（无 module. 前缀）")


# 演示
dp_state = model_dp.state_dict()
model_new = DemoModel()
load_state_dict_flexible(model_new, dp_state)

## 8. GPU 可见性控制

使用 `CUDA_VISIBLE_DEVICES` 环境变量控制程序能看到的 GPU

In [ ]:
# GPU 可见性控制
print("=== CUDA_VISIBLE_DEVICES ===")

# 当前设置
visible = os.environ.get('CUDA_VISIBLE_DEVICES', '未设置')
print(f"当前 CUDA_VISIBLE_DEVICES: {visible}")

# 常见用法（在终端或脚本开头设置）
print("\n常见用法（在 Python 代码最开始设置，或在终端设置）:")
print()
print('# 方式1: 在终端设置（推荐）')
print('# CUDA_VISIBLE_DEVICES=0 python train.py      # 只用 GPU 0')
print('# CUDA_VISIBLE_DEVICES=0,1 python train.py    # 用 GPU 0 和 1')
print('# CUDA_VISIBLE_DEVICES="" python train.py     # 不用任何 GPU')
print()
print('# 方式2: 在代码中设置（必须在 import torch 之前）')
print('# import os')
print('# os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"  # 只用物理 GPU 2 和 3')
print('# import torch  # 之后 torch 只能看到 2 个 GPU')
print()
print("注意:")
print("  设置后，torch.cuda.device_count() 只返回可见的 GPU 数")
print("  GPU 编号从 0 开始重新编号")
print("  例如 CUDA_VISIBLE_DEVICES=2,3 时，物理 GPU2 变为 cuda:0")

## 9. 设备感知的训练代码

编写能自动适应不同设备的训练代码

In [ ]:
import torch.optim as optim


def device_aware_training():
    """设备感知的完整训练示例"""
    # 1. 自动选择设备
    device = get_device()

    # 2. 创建模型
    model = DemoModel(input_dim=20, output_dim=5)

    # 3. 多 GPU 包装（如果有多个 GPU）
    if torch.cuda.device_count() > 1:
        print(f"使用 {torch.cuda.device_count()} 个 GPU")
        model = nn.DataParallel(model)

    # 4. 将模型迁移到设备（Model.to() 是原地操作）
    model.to(device)

    # 5. 创建优化器和损失函数
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    # 6. 创建合成数据
    torch.manual_seed(42)
    X = torch.randn(100, 20)
    y = torch.randint(0, 5, (100,))

    # 7. 训练循环
    model.train()
    for epoch in range(10):
        # 重要：每次都要将数据迁移到正确的设备
        # Tensor.to() 是非原地操作，必须接收返回值！
        X_batch = X.to(device)
        y_batch = y.to(device)

        output = model(X_batch)
        loss = criterion(output, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 5 == 0:
            acc = (output.argmax(1) == y_batch).float().mean()
            print(f"Epoch {epoch+1:3d} | loss={loss:.4f} | acc={acc:.4f}")

    # 8. 保存模型（处理 DataParallel）
    if isinstance(model, nn.DataParallel):
        state_dict = model.module.state_dict()
    else:
        state_dict = model.state_dict()
    print(f"\n模型 state_dict 已准备好保存 ({len(state_dict)} 个参数)")


device_aware_training()

## 10. 总结

### 设备管理
- 使用 `torch.device` 管理设备
- 检查: `torch.cuda.is_available()`, `torch.backends.mps.is_available()`
- 编写设备无关的代码: 使用 `get_device()` 函数

### 数据迁移（最重要的区别）

| 操作 | 是否原地 | 正确用法 |
|------|---------|----------|
| `tensor.to(device)` | 非原地 | `x = x.to(device)` |
| `model.to(device)` | 原地 | `model.to(device)` |

### cuDNN 设置
- `benchmark=True`: 自动搜索最快算法（输入尺寸固定时推荐）
- `deterministic=True`: 保证可重复性

### DataParallel
- 简单的多 GPU 并行: `model = nn.DataParallel(model)`
- 注意 `"module."` 前缀问题
- 保存时使用 `model.module.state_dict()`

### GPU 可见性
- `CUDA_VISIBLE_DEVICES=0,1 python train.py`
- 必须在 `import torch` 之前设置

---

## 练习题

### 基础题
1. 编写一个 `get_device()` 函数，优先使用 CUDA，其次 MPS，最后 CPU，并打印设备信息
2. 创建一个张量和一个模型，分别演示 `.to(device)` 的正确用法

### 进阶题
3. 编写一个完整的设备感知训练脚本：
   - 自动检测设备
   - 支持多 GPU（DataParallel）
   - 正确保存/加载模型（处理 module. 前缀）
   - 在 CPU 上也能正常运行

### 挑战题
4. 实现一个 GPU 内存监控装饰器：
   - 在函数执行前后记录 GPU 内存使用
   - 打印内存变化量
   - 适用于分析哪些操作消耗了大量 GPU 内存

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的更多练习题来巩固知识。